# Pokemon Battle Prediction - FDS 2026

This notebook clones our GitHub repository and uses the `src` package to
reproduce the three selected submissions. All the code lives in the repository;
here we only call it.

Repository: https://github.com/Hiperversum/Pokemon-Challange-ML

In [ ]:
import os, sys

!pip install -q json5

REPO_URL = "https://github.com/Hiperversum/Pokemon-Challange-ML.git"
REPO_DIR = "/kaggle/working/Pokemon-Challange-ML"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
sys.path.insert(0, REPO_DIR)

In [ ]:
from src import config
from src.evaluation.validation import compare_models, cross_validate
from src.models.catalog import build_model
from src.models.presets import build_ensemble, build_tuned_model
from src.features.load_dataset import load_dataset
from src.models.submission import fit_and_predict, make_submission

print("Data folder:", config.get_data_dir())
X_train, y_train, X_test = load_dataset()
print("Train:", X_train.shape, " Test:", X_test.shape)

## Validation

All decisions are taken with stratified 5-fold cross-validation on the training
set. The table below compares the base models on the full feature set.

In [ ]:
base_models = {name: build_model(name) for name in
               ["logreg", "knn", "tree", "random_forest", "adaboost", "hist_gradient_boosting"]}
comparison = compare_models(base_models, X_train.values, y_train.values)
comparison

## Submission 1 - Soft voting of five tuned models

Average of the predicted probabilities of five models with tuned
hyperparameters: L1 logistic regression, gradient boosting, histogram gradient
boosting, our own gradient-descent logistic regression and a random forest.
Combining linear models with tree ensembles reduces the variance of the single
models. It has the best cross-validation accuracy of all the candidates.

In [ ]:
model = build_ensemble("voting_all")
cv = cross_validate(model, X_train.values, y_train.values)
print(f"CV accuracy: {cv['val_acc']:.4f} +/- {cv['val_std']:.4f}")
preds = fit_and_predict(model, X_train, y_train, X_test)
print(make_submission(X_test.index, preds, "submission_1_voting_all.csv"))

## Submission 2 - L1 logistic regression

Logistic regression with an L1 penalty (C = 0.03) on standardized features. The
penalty sets to zero the weights of the features that do not help, so the model
keeps about 250 of the 583 features. It is simple, very fast and as accurate as
the ensembles, and it is very different from the tree models of the other
submissions.

In [ ]:
model = build_tuned_model("logreg")
cv = cross_validate(model, X_train.values, y_train.values)
print(f"CV accuracy: {cv['val_acc']:.4f} +/- {cv['val_std']:.4f}")
preds = fit_and_predict(model, X_train, y_train, X_test)
print(make_submission(X_test.index, preds, "submission_2_logreg_l1.csv"))

## Submission 3 - XGBoost

Gradient boosted trees (XGBoost) with the hyperparameters found by Optuna
(5-fold CV): 650 very shallow trees (depth 2) with strong subsampling. It is the
best single tree-based model and captures non-linear interactions between the
battle state features.

In [ ]:
model = build_tuned_model("xgboost")
cv = cross_validate(model, X_train.values, y_train.values)
print(f"CV accuracy: {cv['val_acc']:.4f} +/- {cv['val_std']:.4f}")
preds = fit_and_predict(model, X_train, y_train, X_test)
print(make_submission(X_test.index, preds, "submission_3_xgboost.csv"))